# NN 1

```mermaid
graph LR
    %% Input Layer Nodes
    x1((X1))
    x2((X2))
    x3((X3))
    x4((X4))
    x5((X5))
    
    %% Bias Node
    bo((Bias bo))

    %% Hidden/Processing Nodes
    Sum((∑ <br> Summing Junction))
    Act((F <br> Activation Function))

    %% Output Node
    y[Output y]

    %% Connections with Weight Labels
    x1 -- w1 --> Sum
    x2 -- w2 --> Sum
    x3 -- w3 --> Sum
    x4 -- w4 --> Sum
    x5 -- w5 --> Sum
    
    %% Bias Connection
    bo --> Sum

    %% Forward Flow
    Sum -- Yin --> Act
    Act --> y

    %% Custom Styling
    style x1 fill:#000,stroke:#333,stroke-width:1px
    style x2 fill:#000,stroke:#333,stroke-width:1px
    style x3 fill:#000,stroke:#333,stroke-width:1px
    style x4 fill:#000,stroke:#333,stroke-width:1px
    style x5 fill:#000,stroke:#333,stroke-width:1px
    style bo fill:#000,stroke:#333,stroke-width:1px
    style Sum fill:#000,stroke:#0288d1,stroke-width:2px
    style Act fill:#000,stroke:#388e3c,stroke-width:2px
    style y fill:#000,stroke:#f57c00,stroke-width:2px
    ```

### Import Libraries

In [15]:
import torch
import torch.nn as nn

### Model Defined

In [16]:
# create model class

class Model(nn.Module):

    def __init__(self,num_features):

        super().__init__() # call parent constructor
        self.linear = nn.Linear(num_features, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self,features):
        out = self.linear(features)
        out = self.sigmoid(out)

        return out

### Model Calling

In [17]:
# create dataset
features = torch.rand(10,5)

# create model
model = Model(features.shape[1])

# call model for forward pass

# INCORRECT (Bypasses hooks, bad practice)
# model.forward(features)

#  CORRECT (Executes hooks, manages graph, standard practice)
model(features)



tensor([[0.3857],
        [0.3738],
        [0.4712],
        [0.4029],
        [0.5043],
        [0.3860],
        [0.4421],
        [0.4028],
        [0.4244],
        [0.4345]], grad_fn=<SigmoidBackward0>)

Using `model(features)` is preferred because it triggers PyTorch's internal `__call__` method, which automatically manages **forward hooks** and **state tracking** required for gradient calculation. Calling `model.forward(features)` directly bypasses these crucial steps, which can break debugging tools and cause silent errors in your network.


In [18]:
# show model weights and bias
print(model.linear.weight)
print()
print(model.linear.bias)

Parameter containing:
tensor([[-0.4098, -0.3952, -0.2451,  0.3325, -0.0005]], requires_grad=True)

Parameter containing:
tensor([0.0319], requires_grad=True)


### Network Visualize

In [19]:
!pip install torchinfo

### Network Visualize

In [20]:
from torchinfo import summary

summary(model, input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Linear: 1-1                            [10, 1]                   6
├─Sigmoid: 1-2                           [10, 1]                   --
Total params: 6
Trainable params: 6
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

# NN 2

```mermaid
graph LR
    %% Input Layer Nodes
    X1((X1))
    X2((X2))
    X3((X3))
    X4((X4))
    X5((X5))
    
    %% Bias Nodes
    b_h((Bias bh))
    b_o((Bias bo))

    %% Hidden Layer Neurons
    subgraph Hidden Layer
        H1((H1))
        H2((H2))
        H3((H3))
    end

    %% Processing Units for Output Layer
    Sum((∑ <br> Summing Junction))
    Act((F <br> Activation Function))

    %% Output
    y[y]

    %% Layer Headers
    subgraph Inputs
        X1
        X2
        X3
        X4
        X5
    end

    subgraph Operation
        Sum
        Act
    end

    subgraph Output
        y
    end

    %% Input to Hidden Connections
    X1 -- w11 --> H1
    X1 -- w12 --> H2
    X1 -- w13 --> H3
    
    X2 -- w21 --> H1
    X2 -- w22 --> H2
    X2 -- w23 --> H3
    
    X3 -- w31 --> H1
    X3 -- w32 --> H2
    X3 -- w33 --> H3
    
    X4 -- w41 --> H1
    X4 -- w42 --> H2
    X4 -- w43 --> H3
    
    X5 -- w51 --> H1
    X5 -- w52 --> H2
    X5 -- w53 --> H3

    %% Hidden Layer Bias
    b_h --> H1
    b_h --> H2
    b_h --> H3

    %% Hidden to Output Connections
    H1 -- wh1 --> Sum
    H2 -- wh2 --> Sum
    H3 -- wh3 --> Sum
    
    %% Output Layer Bias
    b_o --> Sum

    %% Forward Path to Output
    Sum -- Yin --> Act
    Act --> y

    %% Node Styling
    style X1 fill:#000,stroke:#333333,stroke-width:1.5px
    style X2 fill:#000,stroke:#333333,stroke-width:1.5px
    style X3 fill:#000,stroke:#333333,stroke-width:1.5px
    style X4 fill:#000,stroke:#333333,stroke-width:1.5px
    style X5 fill:#000,stroke:#333333,stroke-width:1.5px
    
    style H1 fill:#000,stroke:#333333,stroke-width:1.5px
    style H2 fill:#000,stroke:#333333,stroke-width:1.5px
    style H3 fill:#000,stroke:#333333,stroke-width:1.5px
    
    style b_h fill:#000,stroke:#ffffff,stroke-width:0px
    style b_o fill:#000,stroke:#ffffff,stroke-width:0px
    
    style Sum fill:#000,stroke:#2e7d32,stroke-width:1.5px
    style Act fill:#000,stroke:#2e7d32,stroke-width:1.5px
    style y fill:#000,stroke:#ffffff,stroke-width:0px


### Model Defined

In [21]:
class Model(nn.Module):

    def __init__(self,num_features):

        super().__init__() # call parent constructor
        self.linear1 = nn.Linear(num_features,3)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(3,1)
        self.sigmoid = nn.Sigmoid()


    def forward(self,features):

        out = self.linear1(features)
        out = self.relu(out)
        out = self.linear2(out)
        out = self.sigmoid(out)

        return out

### Model Calling

In [22]:
# create dataset
features = torch.rand(10,5)

# create model
model = Model(features.shape[1])

# call model for forward pass
model(features)



tensor([[0.3829],
        [0.3573],
        [0.3733],
        [0.3380],
        [0.3603],
        [0.3531],
        [0.3694],
        [0.3525],
        [0.3624],
        [0.3525]], grad_fn=<SigmoidBackward0>)

In [23]:
# show model weights and bias
print(model.linear1.weight)
print()
print(model.linear1.bias)
print()
print(model.linear2.weight)
print()
print(model.linear2.bias)

Parameter containing:
tensor([[ 0.1061, -0.0352,  0.0886,  0.0734, -0.3990],
        [ 0.2883, -0.3359, -0.0350,  0.4149, -0.2989],
        [ 0.2362, -0.0523, -0.4007,  0.0270, -0.0295]], requires_grad=True)

Parameter containing:
tensor([ 0.2442, -0.3275,  0.0868], requires_grad=True)

Parameter containing:
tensor([[-0.4939,  0.2828, -0.1706]], requires_grad=True)

Parameter containing:
tensor([-0.4678], requires_grad=True)


### Network Visualize

In [24]:
summary(model, input_size=(10,5))

Layer (type:depth-idx)                   Output Shape              Param #
Model                                    [10, 1]                   --
├─Linear: 1-1                            [10, 3]                   18
├─ReLU: 1-2                              [10, 3]                   --
├─Linear: 1-3                            [10, 1]                   4
├─Sigmoid: 1-4                           [10, 1]                   --
Total params: 22
Trainable params: 22
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

# Sequential Container

# Model Define

Here we define our neural network model by inheriting from nn.Module.

In [27]:
class Model(nn.Module):

  def __init__(self,num_features):

    super().__init__() # call parent constructor
    self.network = nn.Sequential(
        nn.Linear(num_features,3),
        nn.ReLU(),
        nn.Linear(3,1),
        nn.Sigmoid()
    )

  def forward (self, features):

    out = self.network(features)
    return out


### Model Calling

In [28]:
# create dataset
features = torch.rand(10,5)

# create model
model = Model(features.shape[1])

# call model for forward pass
model(features)

tensor([[0.5650],
        [0.5645],
        [0.5744],
        [0.5698],
        [0.5571],
        [0.5578],
        [0.5744],
        [0.5606],
        [0.5744],
        [0.5528]], grad_fn=<SigmoidBackward0>)

In [31]:
print("First layer weights:")
print(model.network[0].weight)

print()

print("First layer biases:")
print(model.network[0].bias)

print()
print("="*50)
print()

print("Third layer weights:")
print(model.network[2].weight)

print()

print("Third layer biases:")
print(model.network[2].bias)


First layer weights:
Parameter containing:
tensor([[-0.1524,  0.2173, -0.1790,  0.0424, -0.3176],
        [-0.0942,  0.2775, -0.1485, -0.2983, -0.0528],
        [ 0.0409, -0.1304, -0.1791, -0.3833, -0.2224]], requires_grad=True)

First layer biases:
Parameter containing:
tensor([ 0.2361, -0.2422, -0.4334], requires_grad=True)


Third layer weights:
Parameter containing:
tensor([[-0.3991,  0.1836,  0.2456]], requires_grad=True)

Third layer biases:
Parameter containing:
tensor([0.2997], requires_grad=True)
